In [ ]:
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt

import torch
import networkx as nx
from torch_geometric.utils import to_networkx

import pickle
import tensorflow as tf

from scipy.spatial.distance import euclidean
from fastdtw import fastdtw
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

import random

In [ ]:
tick_font = {'family': 'serif', 'size': 10}
title_font = {'family': 'serif', 'size': 14}
legend_font = {'family': 'serif', 'size': 10}
labels_font = {'family': 'serif', 'size': 10} #Times New Roman

## LOAD DATA

In [ ]:
# LSTM AE PREDICTIONS
path_lstm = '20251223_bestModel_32_batches_150_epochs'
with open(f'../data/results/ae/{path_lstm}_predictions.pkl', 'rb') as f:
    lstm_pred = pickle.load(f)

# LSTM AE TEST DATASET
with open(f'../data/final_dataset/sequence/test_dataset/ae_test_dataset.pkl', 'rb') as f:
    lstm_test = pickle.load(f)

In [ ]:
# SIAMESE LSTM AE TRAINING DATA
#original_lines = np.load(f'../data/preprocessing/normalized/normalized_local_newsyn_original.npy')
interpolated_lines_original = np.load(f'../data/preprocessing/interpolated/interpolated_lines.npy')

rotated_lines_original = np.load(f'../data/preprocessing/rotated/rotated_lines_newsyn_original.npy')
rotated_lines_close = np.load(f'../data/preprocessing/rotated/rotated_lines_newsyn_close.npy')
rotated_lines_far = np.load(f'../data/preprocessing/rotated/rotated_lines_newsyn_far.npy')

normalized_rotated_lines_original = np.load(f'../data/preprocessing/normalized/normalized_local_newsyn2_original.npy')
normalized_rotated_lines_close = np.load(f'../data/preprocessing/normalized/normalized_local_newsyn2_close.npy')
normalized_rotated_lines_far = np.load(f'../data/preprocessing/normalized/normalized_local_newsyn2_far.npy')

In [ ]:
# SIAMESE LSTM AE PREDICTIONS
path_siamese = '20260119-1907_BO_NewSynData_16_batches_300_epochs'
with open(f'../data/results/siamese/{path_siamese}_predictions.pkl', 'rb') as f:
    pred_data = pickle.load(f)

with open(f'../data/final_dataset/sequence/siamese_test_dataset.pkl', 'rb') as f:
    test_data = pickle.load(f)

siamese_pred_a = pred_data["pred_siamese_a"]
siamese_pred_b = pred_data["pred_siamese_b"]

siamese_test_clean_a = test_data["siamese_test_clean_a"]
siamese_test_clean_b = test_data["siamese_test_clean_b"]
siamese_test_noisy_a = test_data["siamese_test_noisy_a"]
siamese_test_noisy_b = test_data["siamese_test_noisy_b"]

In [ ]:
# SEQUENTIAL GRAPH PREDICTIONS
path_graph_seq = '20260119-1521_newData_Sequential_Huber_1_batches_300_epochs'
with open(f'../data/results/graph_seq/{path_graph_seq}_predictions.pkl', 'rb') as f:
    graph_seq_pred = pickle.load(f)

with open(f'../data/final_dataset/graph/test_dataset/graph_seq_test_data_newsyn.pkl', 'rb') as f:
    graph_seq_test = pickle.load(f)

In [ ]:
# DELAUNAY GRAPH PREDICTIONS
path_graph_delaunay = '20260119-1706_newData_Delaunay_Huber_1_batches_300_epochs'
with open(f'../data/results/graph_del/{path_graph_delaunay}_predictions.pkl', 'rb') as f:
    graph_del_pred = pickle.load(f)

with open(f'../data/final_dataset/graph/test_dataset/graph_del_test_data_newsyn.pkl', 'rb') as f:
    graph_del_test = pickle.load(f)

In [ ]:
graphs_del = torch.load(f'../data/final_dataset/graph/graphs_newsyn_delaunay.pt', weights_only=False)
graphs_seq = torch.load(f'../data/final_dataset/graph/graphs_newsyn_sequential.pt', weights_only=False)

In [ ]:
from matplotlib import pyplot as plt

def plot_input_seq(lines, lines_syn_1=None, lines_syn_2=None, data_range=range(0,1), description=None, saving_img=True):

    for i in data_range:
        
        plt.figure(figsize=(10, 8))
        line = lines[i]
        x, y = line[:, 0], line[:, 1]
        #plt.scatter(x, y, color='#00305D', label='Original Line', s=20)
        plt.plot(x, y, color='#00305D', label=f'Original Line')

        if lines_syn_1 is not None:
            line_s1 = lines_syn_1[i]
            x_s1, y_s1 = line_s1[:, 0], line_s1[:, 1]
            plt.plot(x_s1, y_s1, color='#EC9A29', label=f'Synthetic Original')

        if lines_syn_2 is not None:
            line_s2 = lines_syn_2[i]
            x_s2, y_s2 = line_s2[:, 0], line_s2[:, 1]
            plt.plot(x_s2, y_s2, color='#A8201A', label=f'Synthetic Displaced')

        if description:
            plt.title(f'{description} - Line number {i}', fontdict=title_font)
        else:
            plt.description(f'Line number {i}', fontdict=title_font)
        
        plt.xlabel('X', fontdict=labels_font)
        plt.ylabel('Y', fontdict=labels_font)
        plt.xticks(fontsize=tick_font['size'], family=tick_font['family'])
        plt.yticks(fontsize=tick_font['size'], family=tick_font['family'])
        plt.legend(prop=legend_font)

        plt.axis('equal')
        plt.grid(True)

        if saving_img:
            description = description.replace(" ", "_").lower()
            save_path = f'../data/figures/encodings/encoding_input_sequences_{description}_newsyn.png'
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        
        plt.show()

In [ ]:
data_range = range(0,15)
saving_img=False

# plot_input_seq(interpolated_lines_original, data_range=data_range, description="Original Line", saving_img=saving_img)
# plot_input_seq(rotated_lines_original, data_range=data_range, description="Rotated Line", saving_img=saving_img)
# plot_input_seq(rotated_lines_original, rotated_lines_close, rotated_lines_far, data_range=data_range, description="Original and Synthetic Lines", saving_img=saving_img)
plot_input_seq(normalized_rotated_lines_original, normalized_rotated_lines_close, normalized_rotated_lines_far, data_range=data_range, description="Normalized Lines", saving_img=saving_img)

In [ ]:
def plot_input_graph(data, index, description='', saving_img=False):
    # Convert PyG Data -> NetworkX
    G = to_networkx(data, to_undirected=True)
    pos = {i: (float(data.x[i][1]), float(data.x[i][2])) for i in range(data.num_nodes)}

    # Node colors by line_id
    color_map = ["#143642", "#EC9A29"]
    colors = [color_map[int(data.x[i][0].item())] for i in range(data.num_nodes)]

    plt.figure(figsize=(8, 7))
    plt.title(f'Constructed Graph with {description} Edges - Line {index}', fontdict=title_font)
    plt.xlabel('X', fontdict=labels_font)
    plt.ylabel('Y', fontdict=labels_font)
    #plt.axis('equal')

    # Draw graph
    nx.draw(
        G, pos,
        node_size=5,
        node_color=colors,
        edge_color="#B3B5B6A6",
        alpha=0.8
    )

    # ---- Plot shift vectors as arrows ----
    xy = data.x[:, 1:3].cpu().numpy()
    dxy = data.y.cpu().numpy()

    plt.quiver(
        xy[:, 0], xy[:, 1],          # start points
        dxy[:, 0], dxy[:, 1],        # direction (dx, dy)
        angles='xy', scale_units='xy', scale=1,
        width=0.002, color='#A8201A',
    )

    # ---- Legend ----
    legend_elements = [
        Line2D([0], [0],marker='o',color='none',label='Line A', markerfacecolor=color_map[0], markeredgecolor='none', markersize=6),
        Line2D([0], [0],marker='o',color='none',label='Line B',markerfacecolor=color_map[1], markeredgecolor='none', markersize=6)
    ]
    plt.axis('equal')

    plt.legend(handles=legend_elements, prop=legend_font)

    plt.axis('on')
    plt.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
    
    if saving_img:
        description = description.replace(" ", "_").lower()
        save_path = f'../data/figures/encoding/input_graph_{description}_new.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()


In [ ]:
description_seq = 'Sequential and Cross'
description_del = 'Delaunay and Sequential'

for i in range(14,20):
    plot_input_graph(graphs_del[i], i, description_del, saving_img=False)
    plot_input_graph(graphs_seq[i], i, description_seq, saving_img=False)

## TRAINING LOSSES

In [ ]:
def plot_history_seqbased(history, epochs, description, saving_img=False):
    plt.figure(figsize=(10, 6))
    plt.plot(history['loss'], label='Training Loss', color='#143642')

    if 'val_loss' in history.keys():
        plt.plot(history['val_loss'], label='Validation Loss', color='#EC9A29')

    plt.xlabel('Epochs', fontdict=labels_font)
    plt.ylabel('Loss', fontdict=labels_font)
    plt.legend(prop=legend_font)
    plt.grid(True)
    plt.title(f'LSTM Training & Validation Loss over {epochs} Epochs', fontdict=title_font)

    plt.xticks(fontsize=tick_font['size'], family=tick_font['family'])
    plt.yticks(fontsize=tick_font['size'], family=tick_font['family'])

    if saving_img: 
        description = description.replace(" ", "_").lower()
        save_path = f'../data/figures/losses/train_val_loss_{description}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

In [ ]:
# LSTM AE 
history = np.load(f"../checkpoints/autoencoder/history/{path_lstm}.npy", allow_pickle=True).item()
print(history.keys())
epochs = len(history['loss'])
description='lstm_ae'

plot_history_seqbased(history, epochs, description, False)

In [ ]:
# SIAMESE LSTM AE 
history = np.load(f'../checkpoints/siamese/history/2912_1106_trainable_layers_32_batches_100_epochs.npy', allow_pickle=True).item()
print(history.keys())
epochs = len(history['accuracy'])
description='siamese_lstm_ae'

plot_history_seqbased(history, epochs, description, False)

In [ ]:
def plot_history_graphbased(train_losses, val_losses, description, saving_img=False):
    epochs = len(train_losses)

    plt.figure(figsize=(10,6))
    plt.plot(range(0, epochs + 0), train_losses, label='Train Loss', color='#143642') #range(1, epochs+1)
    plt.plot(range(0, epochs + 0), val_losses, label='Validation Loss', color='#EC9A29') #range(1, epochs+1)
    plt.xlabel('Epoch', fontdict=labels_font)
    plt.ylabel('Loss', fontdict=labels_font)
    plt.title(f'{description} Training & Validation Loss {epochs} over Epochs', fontdict=title_font)
    plt.legend(prop=legend_font)
    plt.xticks(fontsize=tick_font['size'], family=tick_font['family'])
    plt.yticks(fontsize=tick_font['size'], family=tick_font['family'])
    plt.grid(True)

    if saving_img: 
        description = description.replace(" ", "_").lower()
        save_path = f'../data/figures/losses/results_train_val_loss_{description}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

In [ ]:
# GraphSAGE Delaunay
train_losses = np.load(f'../checkpoints/graph/history/{path_graph_delaunay}_train_loss.npy')
val_losses = np.load(f'../checkpoints/graph/history/{path_graph_delaunay}_val_loss.npy')
description='GraphSAGE Delaunay'

plot_history_graphbased(train_losses, val_losses, description, True)

In [ ]:
# GraphSage SEQ
train_losses = np.load(f'../checkpoints/graph/history/{path_graph_seq}_train_loss.npy')
val_losses = np.load(f'../checkpoints/graph/history/{path_graph_seq}_val_loss.npy')
description='GraphSAGE Sequential'

plot_history_graphbased(train_losses, val_losses, description, True)

## PLOTTING PREDICTED LINES

#### PURE LSTM AE

In [ ]:
def plot_lstm_ae_prediction(data_test, data_pred, data_range, description, saving_img=False):

    for i in data_range:
        plt.figure(figsize=(6, 10))
        plt.plot(data_test[i,:,0], data_test[i,:,1], color='#143642', linestyle='--',label='Original')
        plt.plot(data_pred[i,:,0], data_pred[i,:,1], color='#EC9A29',label='Reconstructed')
        plt.legend(prop=legend_font)
        plt.grid(True)
        plt.xlabel('X', fontdict=labels_font)
        plt.ylabel('Y', fontdict=labels_font)
        plt.xticks(fontsize=tick_font['size'], family=tick_font['family'])
        plt.yticks(fontsize=tick_font['size'], family=tick_font['family'])

        plt.axis('equal')
        plt.title(f'LSTM AE - Line {i}', fontdict=title_font)

        if saving_img:
            description = description.replace(" ", "_").lower()
            save_path = f'../data/figures/predictions/results_{description}_pred_line_{i}.png'
            plt.savefig(save_path, dpi=300, bbox_inches='tight')

        plt.show()

In [ ]:
data_range = range(1000,1001)
description = 'lstm_ae'
plot_lstm_ae_prediction(lstm_test, lstm_pred, data_range, description, saving_img=False)

#### SIAMESE LSTM AE

In [ ]:
def plot_siamese_prediction(data_a_test, data_a_pred, data_b_test, data_b_pred, data_b_gt, data_range=range(1,10), description='', saving_img=False):

    for j in data_range:
        plt.figure(figsize=(6, 10))
        
        plt.plot(data_a_test[j,:, 0], data_a_test[j,:, 1], color='#143642', label='Original A', linestyle='--')
        plt.plot(data_b_test[j,:, 0], data_b_test[j,:, 1], color='#EC9A29', label='Original B', linestyle='--')
        plt.plot(data_b_gt[j,:, 0], data_b_gt[j,:, 1], color='#0F8B8D', alpha=0.2, label='Groundtruth B', linestyle='--')
        #plt.plot(data_a_pred[j,:, 0], data_a_pred[j,:, 1], color='#EC9A29', label='Prediction A')
        plt.plot(data_b_pred[j,:, 0], data_b_pred[j,:, 1], color='#A8201A', label='Prediction B')

        plt.title(f'Siamese LSTM AE - Line {j}', fontdict=title_font)
        plt.xlabel('X', fontdict=labels_font)
        plt.ylabel('Y', fontdict=labels_font)

        plt.xticks(fontsize=tick_font['size'], family=tick_font['family'])
        plt.yticks(fontsize=tick_font['size'], family=tick_font['family'])

        plt.legend(prop=legend_font)
        plt.grid(True)
        plt.axis('equal')
        
        if saving_img:
            description = description.replace(" ", "_").lower()
            save_path = f'../data/figures/predictions/siamese_ae/{description}_pred_line_{j}_axis.png'
            plt.savefig(save_path, dpi=300, bbox_inches='tight') 
        plt.show()

In [ ]:
values_b_best = [int(x) for x in random.sample(list(labels_b["best"]), 3)]
values_b_typical = [int(x) for x in random.sample(list(labels_b["typical"]), 3)]
values_b_worst = [int(x) for x in random.sample(list(labels_b["worst"]), 3)]

In [ ]:
data_range = range(2550,2553) # values_b_typical 
#description = 'siamese_b_typical'

plot_siamese_prediction(siamese_test_noisy_a, siamese_pred_a, siamese_test_noisy_b, siamese_pred_b, siamese_test_clean_b, data_range, description, saving_img=False)

#### GRAPHS

In [ ]:
def plot_predictions(type, pred_entry, description, graph_id, saving_img=False):
    coords = pred_entry["coords"]
    true_shift = pred_entry["true_shift"]
    pred_shift = pred_entry["pred_shift"]
    line_id = pred_entry["line_id"]

    orig_a_coords = coords[line_id == 0]
    orig_b_coords = coords[line_id == 1]
    true_b = orig_b_coords + true_shift[line_id == 1]
    pred_b = orig_b_coords + pred_shift[line_id == 1]

    plt.figure(figsize=(10,6)) 

    plt.plot(orig_a_coords[:,0], orig_a_coords[:,1],'-o', color="#143642", markersize=2, label="Original A", linewidth=0.5)
    plt.plot(orig_b_coords[:,0], orig_b_coords[:,1],'-o', color="#EC9A29", markersize=2, label="Original B", linewidth=0.5)
    plt.plot(true_b[:,0], true_b[:,1], '-o', color="#0F8B8D", alpha=0.2, markersize=2, label="Target B", linewidth=0.5)

    # Predicted B
    plt.plot(pred_b[:,0], pred_b[:,1],'-o', color="#A8201A", markersize=2, label="Predicted B", linewidth=0.5)

    # Shift arrows
    for i in range(len(orig_b_coords)):
        plt.arrow(
            orig_b_coords[i,0], orig_b_coords[i,1],
            pred_b[i,0] - orig_b_coords[i,0],
            pred_b[i,1] - orig_b_coords[i,1],
            head_width=0.002, head_length=0.004,
            fc="#A8201A", ec="#A8201A", alpha=0.2
        )

    plt.legend(prop=legend_font)
    plt.title(f"{description} Prediction vs Ground Truth - Graph {graph_id}", fontdict=title_font)
    plt.xlabel("X", fontdict=labels_font)
    plt.ylabel("Y", fontdict=labels_font)
    plt.xticks(fontsize=tick_font['size'], family=tick_font['family'])
    plt.yticks(fontsize=tick_font['size'], family=tick_font['family'])

    plt.axis("equal")

    if saving_img:
        description = description.replace(" ", "_").lower()
        save_path = f'../data/figures/predictions/{type}/{description}_pred_line_{graph_id}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight') 

    plt.show()

#### GRAPH SAGE SEQ

In [ ]:
values_seq_b_best = [int(x) for x in random.sample(list(labels_seq_b["best"]), 10)]

In [ ]:
values_seq_b_typical = [int(x) for x in random.sample(list(labels_seq_b["typical"]), 10)]

In [ ]:
values_seq_b_worst = [int(x) for x in random.sample(list(labels_seq_b["worst"]), 10)]

In [ ]:
description = 'GraphSAGE Sequential Worst B'
type = 'graph_seq'

for graph_id in values_seq_b_worst:
    plot_predictions(type, graph_seq_pred[graph_id], description, graph_id, saving_img=True)


#### GRAPH SAGE DELAUNAY

In [ ]:
# RANDOM SAMPLES OF VALUES
values_del_b_best = [int(x) for x in random.sample(list(labels_del_b["best"]), 10)]

In [ ]:
values_del_b_typical = [int(x) for x in random.sample(list(labels_del_b["typical"]), 10)]

In [ ]:
values_del_b_worst = [int(x) for x in random.sample(list(labels_del_b["worst"]), 10)]

In [ ]:
description = 'GraphSAGE Delaunay Worst B'
type='graph_del'

for graph_id in values_del_b_worst:
    plot_predictions(type, graph_del_pred[graph_id], description, graph_id, saving_img=False)

## METRICS FUNCTIONS

In [ ]:
def discrete_frechet_distance(P, Q):
    n, m = len(P), len(Q)
    ca = -np.ones((n, m))

    def dist(p, q):
        return np.linalg.norm(p - q)

    def recurse(i, j):
        if ca[i, j] > -1:
            return ca[i, j]
        elif i == 0 and j == 0:
            ca[i, j] = dist(P[0], Q[0])
        elif i > 0 and j == 0:
            ca[i, j] = max(recurse(i-1, 0), dist(P[i], Q[0]))
        elif i == 0 and j > 0:
            ca[i, j] = max(recurse(0, j-1), dist(P[0], Q[j]))
        elif i > 0 and j > 0:
            ca[i, j] = max(
                min(
                    recurse(i-1, j),
                    recurse(i, j-1),
                    recurse(i-1, j-1)
                ),
                dist(P[i], Q[j])
            )
        else:
            ca[i, j] = float("inf")
        return ca[i, j]

    return recurse(n-1, m-1)

#NEEDS TO BE CALLED FOR GRAPH 
def discrete_frechet_distance_graph(pred_entry, target_line_id=1):
    coords = pred_entry["coords"]
    true_shift = pred_entry["true_shift"]
    pred_shift = pred_entry["pred_shift"]
    line_id = pred_entry["line_id"]

    syn1_coords = coords[line_id == 0]
    true_s2 = syn1_coords + true_shift[line_id == 0]
    pred_s2 = syn1_coords + pred_shift[line_id == 0]

    return discrete_frechet_distance(true_s2, pred_s2)

In [ ]:
# DTW distance
def dtw_distance(P, Q, plot=False):
    distance, _ = fastdtw(P, Q, dist=euclidean)

    if plot: 
        #PLOT 
        P = np.array(P)
        Q = np.array(Q)
        plt.figure(figsize=(6, 4))
        plt.plot(P[:, 0], P[:, 1], label='true_s2', color='#EC9A29')
        plt.plot(Q[:, 0], Q[:, 1], label='pred_s2', color='#A8201A')

        # DTW alignment as faint gray lines
        for (i, j) in path[::max(len(path)//50, 1)]:  # sample for clarity
            plt.plot([P[i, 0], Q[j, 0]], [P[i, 1], Q[j, 1]], color='gray', alpha=0.3)

        plt.title(f'DTW distance ≈ {distance:.3f}', fontdict=title_font)
        plt.xlabel('x', fontdict=labels_font)
        plt.ylabel('y', fontdict=labels_font)
        plt.legend(prop=legend_font)
        plt.show()

    return distance

#NEEDS TO BE CALLED FOR GRAPH 
def dtw_distance_graph(pred_entry, target_line_id=1, plot=False):
    coords = pred_entry["coords"]
    true_shift = pred_entry["true_shift"]
    pred_shift = pred_entry["pred_shift"]
    line_id = pred_entry["line_id"]

    syn1_coords = coords[line_id == 1]
    true_s2 = syn1_coords + true_shift[line_id == 1]
    pred_s2 = syn1_coords + pred_shift[line_id == 1]

    return dtw_distance(true_s2, pred_s2)

In [ ]:
# DTW similarity (0-1)
def dtw_similarity(P, Q, radius=5):
    distance, path = fastdtw(P, Q, dist=euclidean, radius=radius)
    normalized_dist = distance / len(path)
    similarity = 1.0 / (1.0 + normalized_dist)
    return similarity

# DTW similarity GRAPH 
def dtw_similarity_graph(pred_entry, target_line_id=1):
    coords = pred_entry["coords"]
    true_shift = pred_entry["true_shift"]
    pred_shift = pred_entry["pred_shift"]
    line_id = pred_entry["line_id"]

    syn1_coords = coords[line_id == 0]
    true_s2 = syn1_coords + true_shift[line_id == 0]
    pred_s2 = syn1_coords + pred_shift[line_id == 0]

    return dtw_similarity(true_s2, pred_s2)

In [ ]:
def dtw_avg_distance(P, Q, radius=5):
    distance, path = fastdtw(P, Q, dist=euclidean, radius=radius)
    avg_distance = distance / len(path)
    return avg_distance

In [ ]:
# Area between curves
def area_between_curves(P, Q, plot=False):
    P_sorted = P[np.argsort(P[:, 0])]
    Q_sorted = Q[np.argsort(Q[:, 0])]

    common_x = np.linspace(
        max(P_sorted[0, 0], Q_sorted[0, 0]),
        min(P_sorted[-1, 0], Q_sorted[-1, 0]),
        num=100
    )
    P_interp_y = np.interp(common_x, P_sorted[:, 0], P_sorted[:, 1])
    Q_interp_y = np.interp(common_x, Q_sorted[:, 0], Q_sorted[:, 1])

    area = np.trapezoid(np.abs(P_interp_y - Q_interp_y), x=common_x)

    if plot: 
        #PLOT
        plt.figure(figsize=(6, 4)) 
        plt.plot(common_x, P_interp_y, label='true_s2', color='#EC9A29')
        plt.plot(common_x, Q_interp_y, label='pred_s2', color='#A8201A')
        plt.fill_between(common_x, P_interp_y, Q_interp_y, color='#EC9A29', alpha=0.1)
        plt.title(f'Area between curves ≈ {area:.3f}', fontdict=title_font)
        plt.xlabel('x', fontdict=labels_font)
        plt.ylabel('y', fontdict=labels_font)
        plt.legend(prop=legend_font)
        plt.show()

    return area


def area_between_curves_graph(pred_entry, target_line_id=1, plot=False):
    coords = pred_entry["coords"]
    true_shift = pred_entry["true_shift"]
    pred_shift = pred_entry["pred_shift"]
    line_id = pred_entry["line_id"]

    syn1_coords = coords[line_id == 1]
    true_s2 = syn1_coords + true_shift[line_id == 1]
    pred_s2 = syn1_coords + pred_shift[line_id == 1]

    return area_between_curves(true_s2, pred_s2)

In [ ]:
def quad_area(p1, p2, q1, q2):
    def triangle_area(a, b, c):
        ab = b - a
        ac = c - a
        return 0.5 * abs(ab[0] * ac[1] - ab[1] * ac[0])
        #return 0.5 * abs(np.cross(b - a, c - a))

    return (
        triangle_area(p1, p2, q1) +
        triangle_area(p2, q2, q1)
    )

def quad_area_between_curves(P, Q):
    i = j = 0
    area = 0.0

    while i < len(P) - 1 and j < len(Q) - 1:
        p1, p2 = P[i], P[i + 1]
        q1, q2 = Q[j], Q[j + 1]

        area += quad_area(p1, p2, q1, q2)

        dp = np.linalg.norm(p2 - p1)
        dq = np.linalg.norm(q2 - q1)

        if dp < dq:
            i += 1
        else:
            j += 1

    return area

In [ ]:
def combined_score_fda(frechet_vals, dtw_vals, area_vals):
    frechet = np.array(frechet_vals)
    dtw = np.array(dtw_vals)
    area = np.array(area_vals)

    def minmax(x):
        return (x - x.min()) / (x.max() - x.min())

    frechet_n = minmax(frechet)
    dtw_n = minmax(dtw)
    area_n = minmax(area)

    # Equal-weight combination
    combined = (frechet_n + dtw_n + area_n) / 3.0
    return combined


def classify_by_percentiles(combined_score, low=10, high=90):
    p_low = np.percentile(combined_score, low)
    p_high = np.percentile(combined_score, high)

    best_idx = np.where(combined_score <= p_low)[0]
    typical_idx = np.where((combined_score > p_low) & (combined_score < p_high))[0]
    worst_idx = np.where(combined_score >= p_high)[0]

    return {
        "best": best_idx,
        "typical": typical_idx,
        "worst": worst_idx,
        "p_low": p_low,
        "p_high": p_high
    }

## METRICS CALCULATION & PLOT

In [ ]:
metric_labels = {
    "frechet_line_0": "Fréchet Distance (Line A)",
    "frechet_line_1": "Fréchet Distance (Line B)",
    "dtw_line_0": "Dynamic Time Warping Distance (Line A)",
    "dtw_line_1": "Dynamic Time Warping Distance (Line B)",
    "dtw_similarity_line_0": "DTW Similarity (Line A)",
    "dtw_similarity_line_1": "DTW Similarity (Line B)",
    "area_line_0": "Area Between the Curves (Line A)",
    "area_line_1": "Area Between the Curves (Line B)",
}

In [ ]:
def plot_metric_histogram_sequence(values, title="Metric Histogram", description = '', saving_img=False):
    mean_val = np.mean(values)
    median_val = np.median(values)
    
    plt.figure(figsize=(8,5))
    plt.hist(values, bins=50, edgecolor='#143642', linewidth=0.75, color='white')
    plt.axvline(mean_val, color='#A8201A', linestyle='--', label=f'Mean: {mean_val:.2f}', linewidth=0.95)
    plt.axvline(median_val, color='#EC9A29', linestyle='--', label=f'Median: {median_val:.2f}', linewidth=0.95)
    plt.legend(prop=legend_font)
    plt.xlabel(title, fontdict=labels_font)
    plt.ylabel("Number of Lines", fontdict=labels_font)
    plt.title(f"Histogram of {title}", fontdict=title_font)

    plt.xticks(fontsize=tick_font['size'], family=tick_font['family'])
    plt.yticks(fontsize=tick_font['size'], family=tick_font['family'])
    
    if saving_img:
        save_path = f'../data/figures/eval_metrics/siamese/siamese_pred_{description}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight') 
        
    plt.show()

In [ ]:
def plot_metric_histogram_boxplot_sequence(values, title="Metric", description='', saving_img=False):
    mean_val = np.mean(values)
    median_val = np.median(values)
    
    # Create figure with 2 subplots side by side
    fig, (ax_hist, ax_box) = plt.subplots(
        1, 2, 
        figsize=(10,5), 
        gridspec_kw={"width_ratios": [3, 1]}  # histogram wider than box plot
    )
    
    # Histogram
    ax_hist.hist(values, bins=50, edgecolor='#143642', linewidth=0.75, color='white')
    ax_hist.axvline(mean_val, color='#A8201A', linestyle='--', label=f'Mean: {mean_val:.2f}', linewidth=0.95)
    ax_hist.axvline(median_val, color='#EC9A29', linestyle='--', label=f'Median: {median_val:.2f}', linewidth=0.95)

    ax_hist.set_xlabel(title, fontdict=labels_font)
    ax_hist.set_ylabel("Number of Lines", fontdict=labels_font)
    ax_hist.legend(prop=legend_font)
    
    ax_hist.tick_params(axis='both', labelsize=tick_font['size'])
    ax_hist.set_title(f"Histogram of {title}", fontdict=title_font)
    
    # Box plot on the right
    ax_box.boxplot(values, vert=True, patch_artist=True,
                   boxprops=dict(facecolor='white', color='#143642', linewidth=0.75),
                   medianprops=dict(color='#EC9A29', linewidth=0.95),
                   whiskerprops=dict(color='#143642', linewidth=0.75),
                   capprops=dict(color='#143642', linewidth=0.75))
    ax_box.set_xticks(fontdict=labels_font)
    ax_box.set_ylabel("Values", fontdict=labels_font)
    ax_box.set_title(f"Box Plot", fontdict=title_font)
    ax_box.tick_params(axis='y', labelsize=tick_font['size'])

    plt.setp(ax_hist.get_xticklabels(), **labels_font)
    plt.setp(ax_hist.get_yticklabels(), **labels_font)
    plt.setp(ax_box.get_yticklabels(), **labels_font)
    
    plt.tight_layout()
    
    if saving_img:
        save_path = f'../data/figures/eval_metrics/siamese/siamese_pred_{description}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()

In [ ]:
def plot_metric_histogram_graph(all_predictions, key, type, bins=50, title="Metric", saving_img=False):
    values = [entry[key] for entry in all_predictions]
    mean_val = np.mean(values)
    median_val = np.median(values)
    title = metric_labels.get(key, key.replace("_", " ").title())

    plt.figure(figsize=(8,5))
    plt.hist(values, bins=bins, edgecolor='#143642', linewidth=0.75, color='white')

    plt.axvline(mean_val, color='#A8201A', linestyle='--', label=f'Mean: {mean_val:.2f}', linewidth=0.95)
    plt.axvline(median_val, color='#EC9A29', linestyle='--', label=f'Median: {median_val:.2f}', linewidth=0.95)

    plt.legend(prop=legend_font)
    plt.xlabel(title, fontdict=labels_font)
    plt.ylabel("Number of Lines", fontdict=labels_font)
    plt.title(f"Histogram of {title}", fontdict=title_font)

    if saving_img:
        save_path = f'../data/figures/eval_metrics/{type}/pred_{key}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

In [ ]:
def plot_metric_histogram_boxplot_graph(all_predictions, key, type, bins=50, title="Metric Histogram", saving_img=False):
    values = [entry[key] for entry in all_predictions]
    mean_val = np.mean(values)
    median_val = np.median(values)
    title = metric_labels.get(key, key.replace("_", " ").title())

    # Create figure with 2 subplots side by side
    fig, (ax_hist, ax_box) = plt.subplots(
        1, 2, 
        figsize=(10,5), 
        gridspec_kw={"width_ratios": [3, 1]}  # histogram wider than box plot
    )

    ax_hist.hist(values, bins=bins, edgecolor='#143642', linewidth=0.75, color='white')
    ax_hist.axvline(mean_val, color='#A8201A', linestyle='--', label=f'Mean: {mean_val:.2f}', linewidth=0.95)
    ax_hist.axvline(median_val, color='#EC9A29', linestyle='--', label=f'Median: {median_val:.2f}', linewidth=0.95)

    ax_hist.set_xlabel(title, fontdict=labels_font)
    ax_hist.set_ylabel("Number of Lines", fontdict=labels_font)
    ax_hist.legend(prop=legend_font)

    ax_hist.tick_params(axis='both', labelsize=tick_font['size'])
    ax_hist.set_title(f"Histogram of {title}", fontdict=title_font)

    # Box plot on the right
    ax_box.boxplot(values, vert=True, patch_artist=True,
                   boxprops=dict(facecolor='white', color='#143642', linewidth=0.75),
                   medianprops=dict(color='#EC9A29', linewidth=0.95),
                   whiskerprops=dict(color='#143642', linewidth=0.75),
                   capprops=dict(color='#143642', linewidth=0.75))
    #ax_box.set_xticks(fontdict=labels_font)
    ax_box.set_ylabel("Values", fontdict=labels_font)
    ax_box.set_title(f"Box Plot", fontdict=title_font)
    ax_box.tick_params(axis='y', labelsize=tick_font['size'])

    plt.setp(ax_hist.get_xticklabels(), **labels_font)
    plt.setp(ax_hist.get_yticklabels(), **labels_font)
    plt.setp(ax_box.get_yticklabels(), **labels_font)
    
    plt.tight_layout()

    if saving_img:
        save_path = f'../data/figures/eval_metrics/{type}/pred_{key}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print("saved image!")

    plt.show()

### SEQUENCES

In [ ]:
frechet_vals_a = [discrete_frechet_distance(siamese_test_clean_a[i], siamese_pred_a[i]) for i in range(len(siamese_pred_a))]
dtw_vals_a = [dtw_distance(siamese_test_clean_a[i], siamese_pred_a[i]) for i in range(len(siamese_pred_a))] 
dtw_avg_vals_a = [dtw_avg_distance(siamese_test_clean_a[i], siamese_pred_a[i]) for i in range(len(siamese_pred_a))]
dtw_sim_vals_a = [dtw_similarity(siamese_test_clean_a[i], siamese_pred_a[i]) for i in range(len(siamese_pred_a))]
area_vals_a = [area_between_curves(siamese_test_clean_a[i], siamese_pred_a[i]) for i in range(len(siamese_pred_a))]
quad_area_vals_a = [quad_area_between_curves(siamese_test_clean_a[i], siamese_pred_a[i]) for i in range(len(siamese_pred_a))]

In [ ]:
frechet_vals_b = [discrete_frechet_distance(siamese_test_clean_b[i], siamese_pred_b[i]) for i in range(len(siamese_test_clean_b))]
dtw_vals_b = [dtw_distance(siamese_test_clean_b[i], siamese_pred_b[i]) for i in range(len(siamese_test_clean_b))]
dtw_avg_vals_b = [dtw_avg_distance(siamese_test_clean_b[i], siamese_pred_b[i]) for i in range(len(siamese_test_clean_b))]
dtw_sim_vals_b = [dtw_similarity(siamese_test_clean_b[i], siamese_pred_b[i]) for i in range(len(siamese_test_clean_b))]
area_vals_b = [area_between_curves(siamese_test_clean_b[i], siamese_pred_b[i]) for i in range(len(siamese_test_clean_b))]
quad_area_vals_b = [quad_area_between_curves(siamese_test_clean_b[i], siamese_pred_b[i]) for i in range(len(siamese_test_clean_b))]

In [ ]:
# SAVING METRIC RESULTS 
np.save(f'../data/results/siamese/metrics/frechet_vals_a.npy', frechet_vals_a)
np.save(f'../data/results/siamese/metrics/dtw_vals_a.npy', dtw_vals_a)
np.save(f'../data/results/siamese/metrics/dtw_avg_vals_a.npy', dtw_avg_vals_a)
np.save(f'../data/results/siamese/metrics/dtw_sim_vals_a.npy', dtw_sim_vals_a)
np.save(f'../data/results/siamese/metrics/area_vals_a.npy', area_vals_a)
np.save(f'../data/results/siamese/metrics/quad_area_vals_a.npy', quad_area_vals_a)

np.save(f'../data/results/siamese/metrics/frechet_vals_b.npy', frechet_vals_b)
np.save(f'../data/results/siamese/metrics/dtw_vals_b.npy', dtw_vals_b)
np.save(f'../data/results/siamese/metrics/dtw_avg_vals_b.npy', dtw_avg_vals_b)
np.save(f'../data/results/siamese/metrics/dtw_sim_vals_b.npy', dtw_sim_vals_b)
np.save(f'../data/results/siamese/metrics/area_vals_b.npy', area_vals_b)
np.save(f'../data/results/siamese/metrics/quad_area_vals_b.npy', quad_area_vals_b)


# frechet_vals_a = np.load('../data/results/siamese/metrics/frechet_vals_a.npy', allow_pickle=True)
# dtw_vals_a = np.load('../data/results/siamese/metrics/dtw_vals_a.npy', allow_pickle=True)
# dtw_avg_vals_a = np.load('../data/results/siamese/metrics/dtw_avg_vals_a.npy', allow_pickle=True)
# dtw_sim_vals_a = np.load('../data/results/siamese/metrics/dtw_sim_vals_a.npy', allow_pickle=True)
# area_vals_a = np.load('../data/results/siamese/metrics/area_vals_a.npy', allow_pickle=True)
# quad_area_vals_a = np.load('../data/results/siamese/metrics/quad_area_vals_a.npy', allow_pickle=True)

# frechet_vals_b = np.load('../data/results/siamese/metrics/frechet_vals_b.npy', allow_pickle=True)
# dtw_vals_b = np.load('../data/results/siamese/metrics/dtw_vals_b.npy', allow_pickle=True)
# dtw_avg_vals_b = np.load('../data/results/siamese/metrics/dtw_avg_vals_b.npy', allow_pickle=True)
# dtw_sim_vals_b = np.load('../data/results/siamese/metrics/dtw_sim_vals_b.npy', allow_pickle=True)
# area_vals_b = np.load('../data/results/siamese/metrics/area_vals_b.npy', allow_pickle=True)
# quad_area_vals_b = np.load('../data/results/siamese/metrics/quad_area_vals_b.npy', allow_pickle=True)

In [ ]:
# SIAMESE A AND B 
combined_a = combined_score_fda(frechet_vals_a, dtw_vals_a, area_vals_a)
labels_a = classify_by_percentiles(combined_a)
np.save(f'../data/results/siamese/metrics/combined_a.npy', combined_a)
np.save(f'../data/results/siamese/metrics/labels_a.npy', labels_a)

combined_b = combined_score_fda(frechet_vals_b, dtw_vals_b, area_vals_b)
labels_b = classify_by_percentiles(combined_b)
np.save(f'../data/results/siamese/metrics/combined_b.npy', combined_b)
np.save(f'../data/results/siamese/metrics/labels_b.npy', labels_b)


In [ ]:
print("Best A", labels_a["best"][:10])
print("Best A", labels_a["typical"][:10])
print("Best A", labels_a["worst"][:10])
print("\n")
print("Best B", labels_b["best"][:10])
print("Best B", labels_b["typical"][:10])
print("Best B", labels_b["worst"][:10])

In [ ]:
# plot_metric_histogram_boxplot_sequence(frechet_vals_a, "Discrete Fréchet Distance", "fd_a", True)
# plot_metric_histogram_boxplot_sequence(dtw_vals_a, "DTW Distance", "dtw_dist_a", True)
# plot_metric_histogram_boxplot_sequence(dtw_avg_vals_a, "DTW Average Distance", "dtw_avg_dist_a", True)
# plot_metric_histogram_boxplot_sequence(dtw_sim_vals_a, "DTW Similarity", "dtw_sim_a", True)
# plot_metric_histogram_boxplot_sequence(area_vals_a, "Area Between Curves", "abc_a", True)
# plot_metric_histogram_boxplot_sequence(quad_area_vals_a, "Quad Area Between Curves", "quad_abc_a", True) 

In [ ]:
plot_metric_histogram_boxplot_sequence(frechet_vals_b, "Discrete Fréchet Distance", "fd_b", False)
plot_metric_histogram_boxplot_sequence(dtw_vals_b, "DTW Distance", "dtw_dist_b", False)
plot_metric_histogram_boxplot_sequence(dtw_avg_vals_b, "DTW Average Distance", "dtw_avg_dist_b", False)
plot_metric_histogram_boxplot_sequence(dtw_sim_vals_b, "DTW Similarity", "dtw_sim_b", False)
plot_metric_histogram_boxplot_sequence(area_vals_b, "Area Between Curves", "abc_b", False)
plot_metric_histogram_boxplot_sequence(quad_area_vals_b, "Quad Area Between Curves", "quad_abc_b", False) 

### GRAPH

In [ ]:
metrics_all_graph_seq = []

for pred_entry in graph_seq_pred:
    #print(pred_entry)
    metrics_entry = {}
    for line in [0, 1]:
        metrics_entry[f"frechet_line_{line}"] = discrete_frechet_distance_graph(pred_entry, target_line_id=line)
        metrics_entry[f"dtw_line_{line}"] = dtw_distance_graph(pred_entry, target_line_id=line, plot=False)
        metrics_entry[f"dtw_similarity_line_{line}"] = dtw_similarity_graph(pred_entry, target_line_id=line)
        metrics_entry[f"area_line_{line}"] = area_between_curves_graph(pred_entry, target_line_id=line, plot=False)
    metrics_all_graph_seq.append(metrics_entry)

In [ ]:
metrics_all_graph_del = []

for pred_entry in graph_del_pred:
    #print(pred_entry)
    metrics_entry = {}
    for line in [0, 1]:
        metrics_entry[f"frechet_line_{line}"] = discrete_frechet_distance_graph(pred_entry, target_line_id=line)
        metrics_entry[f"dtw_line_{line}"] = dtw_distance_graph(pred_entry, target_line_id=line, plot=False)
        metrics_entry[f"dtw_similarity_line_{line}"] = dtw_similarity_graph(pred_entry, target_line_id=line)
        metrics_entry[f"area_line_{line}"] = area_between_curves_graph(pred_entry, target_line_id=line, plot=False)
    metrics_all_graph_del.append(metrics_entry)

In [ ]:
np.save("../data/results/graph_del/metrics/metrics_all_graph_del.npy", metrics_all_graph_del, allow_pickle=True)
np.save("../data/results/graph_seq/metrics/metrics_all_graph_seq.npy", metrics_all_graph_seq, allow_pickle=True)

# metrics_all_graph_del = np.load("../data/results/graph_del/metrics/metrics_all_graph_del.npy", allow_pickle=True)
# metrics_all_graph_seq = np.load("../data/results/graph_seq/metrics/metrics_all_graph_seq.npy", allow_pickle=True)

In [ ]:
plot_metric_histogram_boxplot_graph(metrics_all_graph_seq, "frechet_line_1", type='graph_seq', saving_img=True)
plot_metric_histogram_boxplot_graph(metrics_all_graph_seq, "dtw_line_1", type='graph_seq', saving_img=True)
plot_metric_histogram_boxplot_graph(metrics_all_graph_seq, "area_line_1", type='graph_seq', saving_img=True)

In [ ]:
plot_metric_histogram_boxplot_graph(metrics_all_graph_del, "frechet_line_1", type='graph_del', saving_img=True)
plot_metric_histogram_boxplot_graph(metrics_all_graph_del, "dtw_line_1", type='graph_del', saving_img=True)
plot_metric_histogram_boxplot_graph(metrics_all_graph_del, "area_line_1", type='graph_del', saving_img=True)

In [ ]:
# GRAPH SEQ LINE A
frechet_seq_a = [m["frechet_line_0"] for m in metrics_all_graph_seq]
dtw_seq_a = [m["dtw_line_0"] for m in metrics_all_graph_seq]
area_seq_a = [m["area_line_0"] for m in metrics_all_graph_seq]

combined_seq_a = combined_score_fda(frechet_seq_a, dtw_seq_a, area_seq_a)
labels_seq_a = classify_by_percentiles(combined_seq_a)

np.save(f'../data/results/graph_seq/metrics/combined_seq_a.npy', combined_seq_a)
np.save(f'../data/results/graph_seq/metrics/labels_seq_a.npy', labels_seq_a)

# GRAPH SEQ line B
frechet_seq_b = [m["frechet_line_1"] for m in metrics_all_graph_seq]
dtw_seq_b = [m["dtw_line_1"] for m in metrics_all_graph_seq]
area_seq_b = [m["area_line_1"] for m in metrics_all_graph_seq]

combined_seq_b = combined_score_fda(frechet_seq_b, dtw_seq_b, area_seq_b)
labels_seq_b = classify_by_percentiles(combined_seq_b)
np.save(f'../data/results/graph_seq/metrics/combined_seq_b.npy', combined_seq_b)
np.save(f'../data/results/graph_seq/metrics/labels_seq_b.npy', labels_seq_b)


In [ ]:
print("Best A", labels_seq_a["best"][:10])
print("Typical A", labels_seq_a["typical"][:10])
print("Worst A", labels_seq_a["worst"][:10])
print("\n")
print("Best B", labels_seq_b["best"][:10])
print("Typical B", labels_seq_b["typical"][:10])
print("Worst B", labels_seq_b["worst"][:10])

In [ ]:
# GRAPH DEL LINE A
frechet_del_a = [m["frechet_line_0"] for m in metrics_all_graph_del]
dtw_del_a = [m["dtw_line_0"] for m in metrics_all_graph_del]
area_del_a = [m["area_line_0"] for m in metrics_all_graph_del]

combined_del_a = combined_score_fda(frechet_del_a, dtw_del_a, area_del_a)
labels_del_a = classify_by_percentiles(combined_del_a)
np.save(f'../data/results/graph_del/metrics/combined_del_a.npy', combined_del_a)
np.save(f'../data/results/graph_del/metrics/labels_del_a.npy', labels_del_a)

# GRAPH DEL line B
frechet_del_b = [m["frechet_line_1"] for m in metrics_all_graph_del]
dtw_del_b = [m["dtw_line_1"] for m in metrics_all_graph_del]
area_del_b = [m["area_line_1"] for m in metrics_all_graph_del]

combined_del_b = combined_score_fda(frechet_del_b, dtw_del_b, area_del_b)
labels_del_b = classify_by_percentiles(combined_del_b)
np.save(f'../data/results/graph_del/metrics/combined_del_b.npy', combined_del_b)
np.save(f'../data/results/graph_del/metrics/labels_del_b.npy', labels_del_b)


In [ ]:
print("Best A", labels_del_a["best"][:10])
print("Typical A", labels_del_a["typical"][:10])
print("Worst A", labels_del_a["worst"][:10])
print("\n")
print("Best B", labels_del_b["best"][:10])
print("Typical B", labels_del_b["typical"][:10])
print("Worst B", labels_del_b["worst"][:10])